In [1]:
import time
from google.cloud import storage
from google.cloud import bigquery
from google.api_core.exceptions import GoogleAPIError
%load_ext google.cloud.bigquery


# Entity Decomposition

The `movie` entity in the raw dataset contains two types of properties: Properties that change on a daily basis or each time there is a new movie added (e.g. `popularity`, `vote_average`, `vote_count`, etc.) and properties that are relatively static and don't change or change slowly over months or year (e.g. `genre_ids`, `original_language`, `release_date`, etc.).    

In [2]:
%%bigquery
select * from movie_raw.tmdb_movie
limit 5

Query is running:   0%|          |

Downloading:   0%|          |

,movie_id,title,original_language,overview,popularity,release_date,genre_ids,vote_average,vote_count,load_time
0,1160025,Naga,ar,Title: Naga (2023). Stranded amid Riyadh's des...,33.943,9/8/2023,"[53, 35, 18]",5.3,19,2024-02-08 23:34:20.679895+00:00
1,1153366,Head to Head,ar,Title: Head to Head (2023). Hilarity meets dan...,20.481,8/3/2023,"[35, 53, 28]",6.3,59,2024-02-08 23:34:20.679895+00:00
2,1221388,From the Ashes,ar,Title: From the Ashes (2024). At a strict girl...,66.267,1/18/2024,"[53, 18]",6.3,15,2024-02-08 23:34:20.679895+00:00
3,320956,The Cuckold,ar,Title: The Cuckold (2009). The Cuckold is a co...,19.454,8/11/2009,[18],2.8,6,2024-02-08 23:34:20.679895+00:00
4,659676,Masameer: The Movie,ar,"Title: Masameer: The Movie (2020). Danah, a Sa...",46.115,1/9/2020,"[16, 35, 28, 18]",6.1,28,2024-02-08 23:34:20.679895+00:00


In [3]:
%%bigquery
select distinct movie_id,
popularity,
vote_average,
vote_count
from movie_raw.tmdb_movie
order by movie_id, popularity
limit 5

Query is running:   0%|          |

Downloading:   0%|          |

,movie_id,popularity,vote_average,vote_count
0,5,17.694,5.8,2502
1,11,77.825,8.2,19594
2,12,106.343,7.8,18399
3,13,79.889,8.5,26066
4,14,26.600,8.0,11516


In [4]:
%%bigquery
select (select count(*) from movie_raw.tmdb_movie) as total_movie_count,
  (select count(*) from (
    select distinct movie_id
      from movie_raw.tmdb_movie)) as unique_movie_count

Query is running:   0%|          |

Downloading:   0%|          |

,total_movie_count,unique_movie_count
0,9923,9923


Create an movie staging table as an intermediate table. First, convert to correct datatypes

In [11]:
%%bigquery
create or replace table movie_stg.tmdb_movie as
  select *,
    'tmdb' as data_source from movie_stg.tmdb_movie

Query is running:   0%|          |

""


In [6]:
%%bigquery
select * from movie_stg.tmdb_movie
limit 5

Query is running:   0%|          |

Downloading:   0%|          |

,movie_id,title,original_language,popularity,release_date,genre_ids,vote_average,vote_count,load_time,overview
0,1160025,Naga,ar,33.943,9/8/2023,"[53, 35, 18]",5.3,19,2024-02-08 23:34:20.679895+00:00,"Stranded amid Riyadh's desert dunes, Sarah, a ..."
1,1153366,Head to Head,ar,20.481,8/3/2023,"[35, 53, 28]",6.3,59,2024-02-08 23:34:20.679895+00:00,Hilarity meets danger when a lovesick chauffeu...
2,1221388,From the Ashes,ar,66.267,1/18/2024,"[53, 18]",6.3,15,2024-02-08 23:34:20.679895+00:00,"At a strict girls school, a mysterious fire bl..."
3,320956,The Cuckold,ar,19.454,8/11/2009,[18],2.8,6,2024-02-08 23:34:20.679895+00:00,The Cuckold is a controversial dramatic featur...
4,659676,Masameer: The Movie,ar,46.115,1/9/2020,"[16, 35, 28, 18]",6.1,28,2024-02-08 23:34:20.679895+00:00,"Danah, a Saudi girl with a passion for artific..."


Create a person staging table as an intermediate table

# Tmdb_Movie

Now we are ready to create the Tmdb_Movie table which will have a record for each unique movie across time:

In [34]:
%%bigquery
select (select count(*) from movie_raw.tmdb_movie) as total_movie_count,
  (select count(*) from (
    select distinct movie_id
      from movie_raw.tmdb_movie)) as unique_movie_count

Query is running:   0%|          |

Downloading:   0%|          |

,total_movie_count,unique_movie_count
0,29765,9923


In [35]:
%%bigquery
create or replace table movie_stg.tmdb_movie as
    select * EXCEPT(load_time), 'tmdb' as data_source, load_time
    from movie_raw.tmdb_movie

Query is running:   0%|          |

""


# Tmdb_Movie

In [41]:
%%bigquery
create or replace table movie_stg.Tmdb_Movie as
  select distinct movie_id,
    title,
    original_language,
    overview,
    PARSE_DATE('%m/%d/%Y', release_date) AS release_date,
    genre_ids,
    data_source,
    load_time
    from movie_stg.tmdb_movie

Query is running:   0%|          |

""


In [42]:
%%bigquery
select * from movie_stg.Tmdb_Movie limit 5

Query is running:   0%|          |

Downloading:   0%|          |

,movie_id,title,original_language,overview,release_date,genre_ids,data_source,load_time
0,517814,Capernaum,ar,"Title: Capernaum (2018). Zain, a 12-year-old b...",2018-09-20,[18],tmdb,2024-02-24 04:04:01.835961+00:00
1,320956,The Cuckold,ar,Title: The Cuckold (2009). The Cuckold is a co...,2009-08-11,[18],tmdb,2024-02-24 04:04:01.835961+00:00
2,193457,A Cultural Film,ar,Title: A Cultural Film (2000). To watch a porn...,2000-10-04,[35],tmdb,2024-02-24 04:04:01.835961+00:00
3,1221388,From the Ashes,ar,Title: From the Ashes (2024). At a strict girl...,2024-01-18,"[53, 18]",tmdb,2024-02-24 04:04:01.835961+00:00
4,1153366,Head to Head,ar,Title: Head to Head (2023). Hilarity meets dan...,2023-08-03,"[35, 53, 28]",tmdb,2024-02-24 04:04:01.835961+00:00


In [43]:
%%bigquery
select count(*) as unique_movies from movie_stg.Tmdb_Movie

Query is running:   0%|          |

Downloading:   0%|          |

,unique_movies
0,9923


# Movie_History

In [36]:
%%bigquery
select count(*) record_date_not_null
from movie_stg.tmdb_movie
where record_date is not null

Query is running:   0%|          |

Downloading:   0%|          |

,record_date_not_null
0,29765


check to see if there are any field containing only null values

In [37]:
%%bigquery
select count(*) pop_not_null
from movie_stg.tmdb_movie
where popularity is not null

Query is running:   0%|          |

Downloading:   0%|          |

,pop_not_null
0,29765


In [38]:
%%bigquery
select count(*) va_notnull
from movie_stg.tmdb_movie 
where vote_average is not null

Query is running:   0%|          |

Downloading:   0%|          |

,va_notnull
0,29765


In [39]:
%%bigquery
select count(*)  vc_notnull
from movie_stg.tmdb_movie
where vote_count is not null

Query is running:   0%|          |

Downloading:   0%|          |

,vc_notnull
0,29765


In [45]:
%%bigquery
create or replace table movie_stg.Movie_History as
  select DATE(PARSE_TIMESTAMP('%Y-%m-%d %H:%M:%E*S%Ez', CAST(record_date AS STRING))) AS record_date,
    movie_id,
    popularity,
    vote_average,
    vote_count,
    data_source,
    load_time
    from movie_stg.tmdb_movie

Query is running:   0%|          |

""


In [46]:
%%bigquery
select * from movie_stg.Movie_History limit 5

Query is running:   0%|          |

Downloading:   0%|          |

,record_date,movie_id,popularity,vote_average,vote_count,data_source,load_time
0,2024-02-08,1228158,24.199,0.0,0,tmdb,2024-02-24 04:04:01.835961+00:00
1,2024-02-08,840069,15.921,0.0,0,tmdb,2024-02-24 04:04:01.835961+00:00
2,2024-02-08,807590,14.952,0.0,0,tmdb,2024-02-24 04:04:01.835961+00:00
3,2024-02-08,1211067,23.453,0.0,0,tmdb,2024-02-24 04:04:01.835961+00:00
4,2024-02-08,1210466,14.438,0.0,0,tmdb,2024-02-24 04:04:01.835961+00:00


In [47]:
%%bigquery
select (select count(*) from movie_stg.Movie_History) as movie_history_count,
  (select count(*) from movie_stg.tmdb_movie) as original_movie_count

Query is running:   0%|          |

Downloading:   0%|          |

,movie_history_count,original_movie_count
0,29765,29765


# Primary Keys

In [49]:
%%bigquery
alter table movie_stg.Tmdb_Movie
  add primary key (movie_id) not enforced;

Query is running:   0%|          |

""


In [50]:
%%bigquery
select movie_id, count(*) duplicate_records
from movie_stg.Tmdb_Movie
group by movie_id
having count(*) > 1
order by count(*) desc

Query is running:   0%|          |

Downloading: |          |

,movie_id,duplicate_records


In [48]:
%%bigquery
alter table movie_stg.Movie_History
  add primary key (record_date, movie_id) not enforced;

Query is running:   0%|          |

""


In [53]:
%%bigquery
select record_date, movie_id, count(*) duplicate_records
from movie_stg.Movie_History
group by record_date, movie_id
having count(*) > 1


Query is running:   0%|          |

Downloading: |          |

,record_date,movie_id,duplicate_records


# Foreign Keys

In [55]:
%%bigquery
alter table movie_stg.Movie_History add foreign key (movie_id)
    references movie_stg.Tmdb_Movie (movie_id) not enforced;

Query is running:   0%|          |

""


In [56]:
%%bigquery
select count(*) orphan_records
from movie_stg.Movie_History
where movie_id not in (select movie_id from movie_stg.Tmdb_Movie)

Query is running:   0%|          |

Downloading:   0%|          |

,orphan_records
0,0


# Cleanup

Delete the intermediate staging table:

In [36]:
%%bigquery
drop table movie_stg.tmdb_movie

Query is running:   0%|          |

""
